# The Best Neighborhood in Pittsburgh
## Final Ranking

In this notebook, we combine three sub-metrics developed independently by our team members to produce a single, group-level ranking of Pittsburgh neighborhoods. Each sub-metric focuses on a different aspect of "bestness":

- **Ruoyu** — Safety: crime incidents per 100k residents (lower is better)
- **Ruizhi** — Community Activity: normalized community center attendance (higher is better)
- **Zibo** — Neighborhood Vibrancy: parking transaction activity score (higher is better)

Each group member's personal notebook is included in their respective files.

Our goal is to normalize these scores to a common **0~100 scale**, decide on a weighting scheme, and then compute a combined score that allows us to identify the single "best" neighborhood in Pittsburgh according to our data-driven metric.

## The Data

Each metric is reconstructed from its original dataset in this notebook. The individual data cleaning and feature engineering steps for each metric are documented in our three personal notebooks. Here we focus exclusively on **cleaning, normalizing, combining** the three scores and producing a final ranking.

> **Note on coverage:** The Safety dataset covers 81 Pittsburgh neighborhoods; the Community Center dataset covers 13 neighborhoods (limited by facility locations); the Parking dataset covers 9 zone-mapped areas. Because parking zone boundaries do not align with neighborhood boundaries, only 3 neighborhoods appear in all three datasets — too few for a meaningful top-10 ranking. We therefore use **Safety + Community Activity** as our primary combined metric (13 neighborhoods), and treat Parking as supplementary.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Edit DATA_DIR if CSV files are not in the same folder as this notebook
DATA_DIR = '.'

def data(filename):
    return os.path.join(DATA_DIR, filename)

required = ['blotter-data.csv', 'total-population.csv', 'Community_Center_Daily_Attendance.csv']
for f in required:
    status = 'found' if os.path.exists(data(f)) else 'NOT FOUND'
    print(f"{status}: {f}")

## Individual Metric Results

Before combining the metrics, let's first review each team member's individual findings.

### Safety (Ruoyu)

In [ ]:
safety_raw = pd.read_csv(data('blotter-data.csv'))
population_data = pd.read_csv(data('total-population.csv'),
                               usecols=['Neighborhood', 'Estimate; Total'])
population_data.columns = ['neighborhood', 'population']

crime_counts = safety_raw.groupby('INCIDENTNEIGHBORHOOD').size().reset_index()
crime_counts.columns = ['neighborhood', 'crime_count']
crime_counts['neighborhood'] = crime_counts['neighborhood'].replace({
    'Central Northside': 'Central North Side',
    'Troy Hill-Herrs Island': 'Troy Hill',
})
crime_counts = crime_counts[~crime_counts['neighborhood'].isin(
    ['Golden Triangle/Civic Arena', 'Outside City', 'Outside County', 'Outside State']
)]

safety_merged = pd.merge(crime_counts, population_data, on='neighborhood', how='inner')
safety_merged = safety_merged[safety_merged['population'] >= 500]
safety_merged['score'] = (safety_merged['crime_count'] / safety_merged['population']) * 100000

safety_df = safety_merged[['neighborhood', 'score']].copy()

safety_sorted = safety_df.sort_values('score').reset_index(drop=True)
safety_sorted.index = safety_sorted.index + 1
print(safety_sorted.head(10).to_string())

top10_safety = safety_sorted.head(10)
plt.figure(figsize=(10, 6))
plt.bar(top10_safety['neighborhood'], top10_safety['score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Crime Rate per 100k Residents')
plt.title('Top 10 Safest Neighborhoods in Pittsburgh')
plt.tight_layout()
plt.show()

### Community Activity (Ruizhi)

In [ ]:
cc_raw = pd.read_csv(data('Community_Center_Daily_Attendance.csv'))

center_to_neighborhood = {
    'West Penn Community Center':   'Polish Hill',
    'West Penn Fields':             'Polish Hill',
    'West Penn Pool':               'Polish Hill',
    'Arlington Community Center':   'Arlington',
    'Arlington Field':              'Arlington',
    'Magee Community Center':       'Hazelwood',
    'Gladstone Field':              'Hazelwood',
    'Phillips Community Center':    'Homewood South',
    'Phillips Park Field':          'Homewood South',
    'Moore Pool':                   'Homewood South',
    'Warrington Community Center':  'Carrick',
    'Jefferson Community Center':   'Carrick',
    'Jefferson Recreation Center':  'Carrick',
    'Burgwin Field':                'Carrick',
    'Warrington Field':             'Carrick',
    'Paulson Community Center':     'Brookline',
    'Brookline Community Center':   'Brookline',
    'Paulson Field':                'Brookline',
    'Ammon Community Center':       'Bedford Dwellings',
    'Ammon Pool':                   'Bedford Dwellings',
    'Ammon Park':                   'Bedford Dwellings',
    'Ormsby Community Center':      'South Side Flats',
    'Ormsby Field (Playground)':    'South Side Flats',
    'Mellon Tennis Center':         'Shadyside',
    'Schenley Ice Rink':            'Squirrel Hill South',
    'Frazier Park':                 'Greenfield',
    'Frick Environmental Center':   'Point Breeze',
    'Highland Pool':                'Highland Park',
}

cc_raw['neighborhood'] = cc_raw['center_name'].map(center_to_neighborhood)
cc_grouped = cc_raw.groupby('neighborhood')['attendance_count'].sum().reset_index()
cc_grouped.columns = ['neighborhood', 'score']

community_df = cc_grouped.copy()

community_sorted = community_df.sort_values('score', ascending=False).reset_index(drop=True)
community_sorted.index = community_sorted.index + 1
print(community_sorted.head(10).to_string())

top10_cc = community_sorted.head(10)
plt.figure(figsize=(10, 6))
plt.bar(top10_cc['neighborhood'], top10_cc['score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Total Attendance Count')
plt.title('Top 10 Neighborhoods by Community Center Attendance')
plt.tight_layout()
plt.show()

### Neighborhood Vibrancy — Parking Activity (Zibo)

Zibo's metric uses aggregated parking transactions as a proxy for neighborhood vibrancy. Because the parking dataset uses zone-level boundaries (not neighborhood boundaries), scores are mapped manually to the 9 most representative neighborhoods.

In [ ]:
parking_df = pd.DataFrame({
    'neighborhood': [
        'South Side Flats', 'South Side Slopes',
        'Central Business District', 'Strip District',
        'Oakland', 'Shadyside', 'Squirrel Hill South',
        'Lawrenceville', 'North Shore',
    ],
    'score': [1.00, 1.00, 0.85, 0.60, 0.55, 0.45, 0.35, 0.30, 0.25],
})

parking_sorted = parking_df.sort_values('score', ascending=False).reset_index(drop=True)
parking_sorted.index = parking_sorted.index + 1
print(parking_sorted.to_string())

plt.figure(figsize=(10, 6))
plt.bar(parking_sorted['neighborhood'], parking_sorted['score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Parking Activity Score (normalized)')
plt.title('Neighborhood Vibrancy by Parking Activity')
plt.tight_layout()
plt.show()

## Combined Metric Process

Now that we've reviewed each individual metric, we'll combine them into a single comprehensive ranking.

### 1. Clean Data

Before we can combine the metrics, we need to ensure the data is clean and consistent. We keep only neighborhoods that appear in both the Safety and Community datasets to ensure fair comparison, and remove any rows with missing data.

In [ ]:
# Rename score columns
safety_df2    = safety_df.rename(columns={'score': 'safety_score'})
community_df2 = community_df.rename(columns={'score': 'community_score'})
parking_df2   = parking_df.rename(columns={'score': 'parking_score'})

# Keep only neighborhoods present in both Safety and Community
merged = safety_df2.merge(community_df2, on='neighborhood', how='inner')
merged = merged.dropna()

print(f"Neighborhoods appearing in Safety + Community datasets: {len(merged)}")
print(merged.head(10).to_string(index=False))

### Handle Outliers in Safety Score

Some neighborhoods have extremely high crime rates that could distort our normalization process. We cap the safety scores at the **96th percentile** to prevent a few extreme outliers from compressing the scores of the majority of neighborhoods into a narrow range, while still preserving relative differences.

In [ ]:
percentile_96 = merged['safety_score'].quantile(0.96)
print(f"96th percentile of safety score: {percentile_96:.2f}")
print(f"Neighborhoods above 96th percentile: {(merged['safety_score'] > percentile_96).sum()}")

merged['safety_score_capped'] = merged['safety_score'].clip(upper=percentile_96)

### 2. Normalize Scores to [0, 100]

We normalize all scores to a common scale of 0 to 100. This ensures that each metric contributes proportionally to the final score, regardless of their original units or ranges.

In [ ]:
def normalize(series):
    return ((series - series.min()) / (series.max() - series.min())) * 100

merged['safety_norm']    = normalize(merged['safety_score_capped'])  # 0=safest, 100=most dangerous
merged['community_norm'] = normalize(merged['community_score'])       # 0=least active, 100=most active

print(merged[['neighborhood', 'safety_norm', 'community_norm']].to_string(index=False))

### 3.1 Calculate Final Score

Next we can obtain the final score with our combined metric:

**Weights:**
- Community Activity: **1**
- Safety: **2**

**Formula:**

$$\text{Final Score} = \text{Community} - 2 \times \text{Safety}$$

*Note: Higher Safety Score means higher crime rate, so we subtract it.*

In [ ]:
# Calculate final score with weights: Safety = 2 (subtracted), Community = 1
merged['final_score'] = merged['community_norm'] - 2 * merged['safety_norm']

result = merged.sort_values('final_score', ascending=False).reset_index(drop=True)
result['rank'] = range(1, len(result) + 1)

print("TOP 10 BEST NEIGHBORHOODS IN PITTSBURGH")
print(result[['rank', 'neighborhood', 'final_score',
              'safety_norm', 'community_norm']].head(10).to_string(index=False))

### 3.2 Visualization: Top 10 Neighborhoods by Final Score

To better communicate our results, we plot the combined final score for the top 10 neighborhoods.

In [ ]:
top10 = result.head(10)

plt.figure(figsize=(10, 6))
plt.bar(top10['neighborhood'], top10['final_score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Final Score')
plt.title('Top 10 Pittsburgh Neighborhoods by Combined Score\n(Community − 2 × Safety)')
plt.tight_layout()
plt.savefig('combined_top10.png', dpi=150, bbox_inches='tight')
plt.show()

### Supplementary: Three-Metric Score (where parking data is available)

When parking data is also available, we extend the formula:

$$\text{Final Score} = \text{Community} + \text{Parking} - 2 \times \text{Safety}$$

Only 3 neighborhoods appear in all three datasets due to the coarse zone coverage of the parking data.

In [ ]:
# Normalize parking to [0, 100] for consistency
parking_df2['parking_norm'] = parking_df2['parking_score'] * 100

merged_all3 = merged.merge(parking_df2[['neighborhood', 'parking_norm']],
                           on='neighborhood', how='inner')
merged_all3['final_score_3'] = (
    merged_all3['community_norm']
    + merged_all3['parking_norm']
    - 2 * merged_all3['safety_norm']
)

result_all3 = merged_all3.sort_values('final_score_3', ascending=False).reset_index(drop=True)
result_all3['rank'] = range(1, len(result_all3) + 1)

print(f"Neighborhoods in all 3 datasets: {len(result_all3)}")
print(result_all3[['rank', 'neighborhood', 'safety_norm',
                    'community_norm', 'parking_norm', 'final_score_3']].to_string(index=False))

## 4.1 Conclusion

Based on our combined metric (Community Activity − 2 × Safety), **Brookline** emerges as the best neighborhood in Pittsburgh among the 13 neighborhoods that appear in both datasets.

From a data perspective, Brookline performs consistently well across both dimensions:
- It has the **highest community center attendance** in the city (normalized score = 100).
- Its **crime rate is very low**, placing it among the safest residential neighborhoods.

The two runners-up — **Polish Hill** and **Carrick** — also demonstrate strong balanced performance across both safety and community activity.

## 4.2 Reflection

**Ruoyu:** My favorite neighborhood is Squirrel Hill South, where I currently live. Life here is very convenient — there are large supermarkets like Giant Eagle, Asian markets like Panda Supermarket, and many international restaurants. Beyond food and shopping, Squirrel Hill South is also very safe; I often come home late at night and always feel secure. However, Squirrel Hill South ranks lower in our combined metric because its community center attendance is minimal, which drags down its community score. This highlights how a data-driven metric may not fully capture the lived experience of a neighborhood.

**Ruizhi:** Based on my analysis, Brookline is identified as the best neighborhood. But in terms of traffic convenience, Brookline may not be ideal for everyone. My personal preference would depend more on commute access than attendance numbers.

**Zibo:** South Side Flats and South Side Slopes rank highest in parking activity, reflecting their density of restaurants, bars, and entertainment along East Carson Street. Personally, however, I would prefer a neighborhood that balances vibrancy with safety and residential comfort.